# Day 3 Final — Baseline ML: Vietnamese Price Prediction

**Dataset:** `SeanSunny/items_tv_v6` — lọc giá <= 1,000,000 VND  
**Split:** 85,727 train | 3,926 val | 3,872 test  
**Metric chính:** RMSLE (Root Mean Squared Logarithmic Error) — chuẩn Kaggle cho price prediction  
**Best kết quả:** Blended RMSLE = **0.5088** (3,872 test items)

---

## Cấu trúc notebook

| Tier | Model | Mục đích |
|------|-------|----------|
| **1** | Random / Mean / Median | Baseline thống kê — không cần dữ liệu text |
| **2** | LR + Arch A / LR + Arch B | Minh họa tại sao KHÔNG log-transform → RMSLE thảm họa |
| **3** | Ridge / LightGBM / LGB Tuned | ML chính — log1p + category feature |
| **4** | Weighted Blend | Kết hợp top models |

**Key techniques:**
- **Log-transform target:** train trên `log1p(price)`, predict `expm1(pred)` — tác động lớn nhất (-66% RMSLE vs LR raw)
- **Underthesea word segmentation:** tách từ tiếng Việt `"điện_thoại"`, `"màn_hình"` trước TF-IDF
- **Category feature:** one-hot 8 categories + TF-IDF (scipy sparse hstack)
- **Arch C:** kết hợp word bigram + char_wb 3-5gram — bắt brand names, model numbers, specs


In [2]:
2

2

In [3]:
import sys
from pathlib import Path

# pricer_vi nằm ở thư mục cha (Data_processing_for_Vietnamese_data/)
sys.path.insert(0, str(Path().resolve().parent))

import random
import time
import pickle

import numpy as np
import pandas as pd
import plotly.express as px
from scipy.sparse import hstack
from scipy.optimize import minimize as scipy_minimize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb
from tqdm.auto import tqdm

from pricer_vi.items import Item
from pricer_vi.evaluator import rmsle, plot_predictions

SEED = 42
DATASET = "SeanSunny/items_tv_v6"
PRICE_THRESHOLD = 1_000_000
CACHE_DIR = Path().resolve()  # day3/
PLOT_SIZE = 200               # số điểm hiển thị trên chart (metric tính trên full test set)

random.seed(SEED)
np.random.seed(SEED)

print("Setup done.")

Setup done.


## 1. Load Data

Load từ HuggingFace Hub lần đầu, cache thành `.pkl` để lần sau chạy nhanh hơn.  
Filter giá `<= 1,000,000 VND` — loại bỏ 22.1% outlier đắt tiền, giảm skewness từ 6.85 → 1.23.

In [4]:
items_train_cache = CACHE_DIR / "items_train.pkl"
items_val_cache   = CACHE_DIR / "items_val.pkl"
items_test_cache  = CACHE_DIR / "items_test.pkl"

if items_train_cache.exists():
    print("Loading items from local cache...")
    with open(items_train_cache, "rb") as f: train_raw = pickle.load(f)
    with open(items_val_cache,   "rb") as f: val_raw   = pickle.load(f)
    with open(items_test_cache,  "rb") as f: test_raw  = pickle.load(f)
else:
    print(f"Downloading from HuggingFace: {DATASET}")
    train_raw, val_raw, test_raw = Item.from_hub(DATASET)
    with open(items_train_cache, "wb") as f: pickle.dump(train_raw, f)
    with open(items_val_cache,   "wb") as f: pickle.dump(val_raw, f)
    with open(items_test_cache,  "wb") as f: pickle.dump(test_raw, f)
    print("Cached items to disk.")

print(f"Raw: {len(train_raw):,} train | {len(val_raw):,} val | {len(test_raw):,} test")

# Filter <= 1M VND
train = [item for item in train_raw if item.price <= PRICE_THRESHOLD]
val   = [item for item in val_raw   if item.price <= PRICE_THRESHOLD]
test  = [item for item in test_raw  if item.price <= PRICE_THRESHOLD]

print(f"Filtered <= {PRICE_THRESHOLD:,} VND:")
print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

# Numpy arrays hay dùng
train_prices = np.array([item.price for item in train], dtype=float)
val_prices   = np.array([item.price for item in val],   dtype=float)
test_prices  = np.array([item.price for item in test],  dtype=float)
test_names   = [item.title for item in test]

print(f"  Price range: {train_prices.min():,.0f} - {train_prices.max():,.0f} VND")
print(f"  Mean: {train_prices.mean():,.0f} | Median: {np.median(train_prices):,.0f}")

Loading items from local cache...
Raw: 110,000 train | 5,000 val | 5,000 test
Filtered <= 1,000,000 VND:
  Train: 85,727 | Val: 3,926 | Test: 3,872
  Price range: 4,900 - 1,000,000 VND
  Mean: 301,687 | Median: 229,000


## 2. EDA — Exploratory Data Analysis

In [5]:
# --- Price distribution ---
df_price = pd.DataFrame({"price": train_prices})
fig = px.histogram(
    df_price, x="price", nbins=60,
    title="Phân phối giá sản phẩm (Train, <= 1M VND)",
    labels={"price": "Giá (VND)"},
    width=800, height=400,
)
fig.update_xaxes(tickformat=",.0f")
fig.show()

# --- Category distribution ---
cat_counts = pd.Series([item.category for item in train]).value_counts().reset_index()
cat_counts.columns = ["category", "count"]
fig2 = px.bar(
    cat_counts, x="count", y="category", orientation="h",
    title="Số lượng sản phẩm theo danh mục (Train)",
    labels={"count": "Số sản phẩm", "category": ""},
    width=800, height=400,
)
fig2.show()

# --- Sample items ---
print("\nSample items:")
for item in train[:3]:
    print(f"  [{item.category}] {item.title[:60]} | {item.price:,} VND")
    print(f"    Summary: {item.summary[:100]}...")


Sample items:
  [Thời Trang] Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ | 339,000 VND
    Summary: Tiêu đề: Áo len hoodie dày ấm cho nữ  
Danh mục: Thời trang nữ  
Thương hiệu: LiLiLa  
Mô tả: Hoodie...
  [Bách Hóa] Trà lá xanh hương lá dứa Trần Quang (gói 500gr) | 78,000 VND
    Summary: Tiêu đề: Trà lá xanh hương lá dứa Trần Quang (gói 500gr)  
Danh mục: Trà & Đồ uống  
Thương hiệu: Tr...
  [Làm Đẹp - Sức Khỏe] Combo 2 Sữa Rửa Mặt innisfree Kiểm Soát Nhờn Tro Núi Lửa & B | 295,000 VND
    Summary: Tiêu đề: Combo 2 Sữa Rửa Mặt Innisfree Volcanic Pore BHA 150g  
Danh mục: Sữa Rửa Mặt & Chăm sóc da ...


## 3. Text Tokenization — Underthesea

Tiếng Việt cần tách từ trước khi đưa vào TF-IDF. `underthesea.word_tokenize()` nhận diện từ ghép:

| Raw | Sau tokenize |
|-----|-------------|
| `điện thoại thông minh` | `điện_thoại thông_minh` |
| `máy tính xách tay` | `máy_tính xách_tay` |
| `128gb pin 5000mah` | `128gb pin 5000mah` |

**Quan trọng:** underthesea KHÔNG thread-safe → phải pre-tokenize 1 lần và cache `.pkl`.  
Cache đã có sẵn từ run trước — load trực tiếp.

In [6]:
from underthesea import word_tokenize
from multiprocessing import Pool

def tokenize_one(text):
    return word_tokenize(text, format="text")

def load_or_tokenize(cache_path, texts, desc):
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            result = pickle.load(f)
        print(f"Loaded cache: {cache_path.name} ({len(result):,} docs)")
        return result
    print(f"Tokenizing {len(texts):,} {desc} docs...")
    with Pool(4) as p:
        result = list(tqdm(p.imap(tokenize_one, texts, chunksize=500), total=len(texts)))
    with open(cache_path, "wb") as f:
        pickle.dump(result, f)
    print(f"Saved: {cache_path.name}")
    return result

tokenized_train = load_or_tokenize(
    CACHE_DIR / "tokenized_train_1m.pkl",
    [item.summary for item in train], "train"
)
tokenized_val = load_or_tokenize(
    CACHE_DIR / "tokenized_val_1m.pkl",
    [item.summary for item in val], "val"
)
tokenized_test = load_or_tokenize(
    CACHE_DIR / "tokenized_test_1m.pkl",
    [item.summary for item in test], "test"
)

print(f"\nSample raw:       {train[0].summary[:80]}")
print(f"Sample tokenized: {tokenized_train[0][:80]}")

Loaded cache: tokenized_train_1m.pkl (85,727 docs)
Loaded cache: tokenized_val_1m.pkl (3,926 docs)
Loaded cache: tokenized_test_1m.pkl (3,872 docs)

Sample raw:       Tiêu đề: Áo len hoodie dày ấm cho nữ  
Danh mục: Thời trang nữ  
Thương hiệu: Li
Sample tokenized: Tiêu_đề : Áo len_hoodie dày ấm cho nữ Danh_mục : Thời_trang nữ Thương_hiệu : LiL


## 4. Feature Engineering

### 4a. Text Vectorization — 3 Architectures

| Arch | Mô tả | max_features | Đặc điểm |
|------|-------|-------------|----------|
| **A** | TF-IDF raw text, ngram=(1,2), không tách từ | 10,000 | Đơn giản nhất |
| **B** | Underthesea word_tokenize → TF-IDF word unigram | 10,000 | Hiểu từ ghép tiếng Việt |
| **C** | Underthesea → FeatureUnion(word bigram 5K + char_wb 3-5gram 5K) | 10,000 | Bắt specs, brand, model number |

**char_wb** (character whitespace-bounded): bắt patterns như `"128gb"`, `"5000mah"`, `"samsung"` dù text bị viết tắt.

### 4b. Category Feature

8 danh mục → **OneHotEncoder** → 8 sparse features → `scipy.sparse.hstack` với TF-IDF matrix.

**Arch B + Cat** = `TF-IDF_B (10,000) + OneHot_Category (8)` = **10,008 features**  
**Arch C + Cat** = `TF-IDF_C (10,000) + OneHot_Category (8)` = **10,008 features**

### 4c. Log-transform Target

```
y_train = log1p(price)  →  train model  →  predict  →  expm1(pred) = price_vnd
```

RMSLE = sqrt(mean((log1p(pred) - log1p(true))²)) — metric này **đã ở log-space**.  
Train trực tiếp trên `log1p(price)` giúp model tối ưu đúng metric, tránh predict âm.

In [7]:
# === Architecture A: TF-IDF raw text (bigram, no word segmentation) ===
t0 = time.time()
vectorizer_a = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))
train_docs_raw = [item.summary for item in train]
test_docs_raw  = [item.summary for item in test]
X_a_train = vectorizer_a.fit_transform(train_docs_raw)
X_a_test  = vectorizer_a.transform(test_docs_raw)
print(f"Arch A: {X_a_train.shape} ({time.time()-t0:.1f}s)")

# === Architecture B: Underthesea + TF-IDF word unigram ===
t0 = time.time()
vectorizer_b = TfidfVectorizer(max_features=10_000)
X_b_train = vectorizer_b.fit_transform(tokenized_train)
X_b_test  = vectorizer_b.transform(tokenized_test)
X_b_val   = vectorizer_b.transform(tokenized_val)
print(f"Arch B: {X_b_train.shape} ({time.time()-t0:.1f}s)")

# === Architecture C: word bigram + char_wb 3-5gram (FeatureUnion) ===
t0 = time.time()
arch_c = FeatureUnion([
    ("word", TfidfVectorizer(analyzer="word",    ngram_range=(1, 2), max_features=5_000)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=5_000)),
])
X_c_train = arch_c.fit_transform(tokenized_train)
X_c_test  = arch_c.transform(tokenized_test)
X_c_val   = arch_c.transform(tokenized_val)
print(f"Arch C: {X_c_train.shape} ({time.time()-t0:.1f}s)")

# === Category one-hot ===
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat_train = cat_encoder.fit_transform([[item.category] for item in train])
X_cat_val   = cat_encoder.transform([[item.category] for item in val])
X_cat_test  = cat_encoder.transform([[item.category] for item in test])
categories  = list(cat_encoder.categories_[0])
print(f"Category: {X_cat_train.shape[1]} features — {categories}")

# === Combined matrices (text + category) ===
X_bc_train = hstack([X_b_train, X_cat_train])
X_bc_test  = hstack([X_b_test,  X_cat_test])
X_bc_val   = hstack([X_b_val,   X_cat_val])

X_cc_train = hstack([X_c_train, X_cat_train])
X_cc_test  = hstack([X_c_test,  X_cat_test])
X_cc_val   = hstack([X_c_val,   X_cat_val])

print(f"Arch B+Cat: {X_bc_train.shape}")
print(f"Arch C+Cat: {X_cc_train.shape}")

# === Log-transform target ===
log_train_prices = np.log1p(train_prices)
log_val_prices   = np.log1p(val_prices)
print(f"\nLog price: {log_train_prices.min():.2f} - {log_train_prices.max():.2f}")
print(f"  Mean: {log_train_prices.mean():.2f} | Median: {np.median(log_train_prices):.2f}")

Arch A: (85727, 10000) (7.6s)
Arch B: (85727, 10000) (2.8s)
Arch C: (85727, 10000) (30.2s)
Category: 8 features — ['Bách Hóa', 'Làm Đẹp - Sức Khỏe', 'Mẹ và Bé', 'Nhà Cửa - Đời Sống', 'Thời Trang', 'Ô Tô - Xe Máy', 'Điện Lạnh và Gia Dụng', 'Điện Tử - Công Nghệ']
Arch B+Cat: (85727, 10008)
Arch C+Cat: (85727, 10008)

Log price: 8.50 - 13.82
  Mean: 12.34 | Median: 12.34


## 5. Tier 1 — Statistical Baselines

Không dùng text, không dùng ML — chỉ dựa vào thống kê phân phối giá train set.

| Model | Dự đoán | RMSLE thực tế | Nhận xét |
|-------|---------|--------------|----------|
| **Random** | Số ngẫu nhiên trong [4,900 – 1,000,000] | **1.2970** | Tệ nhưng không thảm họa — KHÔNG bao giờ predict 0 |
| **Mean** | 301,687 VND | **0.8219** | Cải thiện khi bỏ variance, nhưng MAPE 106% |
| **Median** | 229,000 VND | **0.7736** | Tốt nhất Tier 1 — robust với outlier |

> Chart: 200 điểm mẫu để quan sát. Metrics (RMSLE, MAE, MAPE, R2) tính trên toàn bộ 3,872 test items.


In [8]:
results = {}  # thu thập kết quả tất cả model

# --- Random ---
min_price, max_price = int(train_prices.min()), int(train_prices.max())
rng = np.random.default_rng(SEED)
pred_random = rng.integers(min_price, max_price + 1, size=len(test)).astype(float)
results["Random"] = plot_predictions(
    test_prices, pred_random,
    title="Random",
    names=test_names, plot_size=PLOT_SIZE,
)

# --- Mean ---
pred_mean = np.full(len(test), train_prices.mean())
results["Mean"] = plot_predictions(
    test_prices, pred_mean,
    title=f"Mean (= {train_prices.mean():,.0f} VND)",
    names=test_names, plot_size=PLOT_SIZE,
)

# --- Median ---
pred_median = np.full(len(test), np.median(train_prices))
results["Median"] = plot_predictions(
    test_prices, pred_median,
    title=f"Median (= {np.median(train_prices):,.0f} VND)",
    names=test_names, plot_size=PLOT_SIZE,
)

## 6. Tier 2 — NLP Baseline: LR + TF-IDF (không log-transform)

Linear Regression train trực tiếp trên giá VND (raw). Đây là cách **sai** — mục đích là minh họa vấn đề.

### Kết quả thực tế — nghịch lý đáng chú ý

| Model | RMSLE | MAE (VND) | MAPE | R2 |
|-------|------:|----------:|-----:|---:|
| **LR + Arch A** | **1.5216** | 124,523 | 63.7% | 45.4% |
| **LR + Arch B** | **1.4950** | 122,215 | 62.2% | 47.0% |
| Random (tier 1) | 1.2970 | 340,331 | 226.0% | -232.1% |

**LR có MAE tốt hơn Random (122K vs 340K), R2 tốt hơn (47% vs -232%), nhưng RMSLE lại TỆ HƠN cả Random.**

---

### Tại sao LR + Arch B (RMSLE = 1.495) tệ hơn cả Random (RMSLE = 1.297)?

**Nguyên nhân gốc rễ: Clipping về 0 + RMSLE là log-scale metric.**

LR tối ưu MSE trên raw VND (301K mean) → tập trung vào sản phẩm đắt.  
Sản phẩm rẻ (< 50K VND) thường bị predict âm → `np.clip(pred, 0, None)` → pred = **0**.

Khi pred = 0, RMSLE đo:
```
error² = (log1p(0) - log1p(true))²  =  (0 - log1p(true))²

Sản phẩm 10,000 VND bị predict 0:
  error² = (0 - log1p(10000))² = (0 - 9.21)² = 84.8   ← thảm họa

Cùng sản phẩm, Random predict 300,000 VND:
  error² = (log1p(300000) - log1p(10000))² = (12.61 - 9.21)² = 11.6   ← chấp nhận được
```

LR clip = 0 tệ hơn Random **7.3x** trên sản phẩm rẻ trong RMSLE space.

---

### Tại sao R2 = 47% vẫn tốt dù RMSLE thảm họa?

R2 và RMSLE đo hai thứ khác nhau:

| Metric | Đo cái gì | Đơn vị | LR "nhìn" vào đâu |
|--------|----------|--------|-------------------|
| **R2** | Sai số tuyệt đối (VND)² | VND | Sản phẩm đắt (variance cao) |
| **RMSLE** | Sai số tương đối (%) đồng đều mọi giá | Log | Mọi price range bình đẳng |

- LR dự đoán khá tốt cho sản phẩm 200K–1M VND → R2 và MAE đẹp
- Sản phẩm rẻ (5K–50K): predict âm → clip 0 → sai số VND nhỏ (10K), nhưng RMSLE khổng lồ
- R2 **không thấy** thảm họa ở sản phẩm rẻ vì sai số 10K VND nhỏ so với variance tổng
- RMSLE **thấy ngay** vì nó đo trên log-scale — 10K VND sai = cùng penalty như 1M VND sai 100x

> **Kết luận:** Khi metric là RMSLE, log-transform target là **bắt buộc**.
> Predict 0 cho bất kỳ sản phẩm nào là sai lầm chết người — tệ hơn đoán mò.


In [9]:
# === LR + Arch A (raw text bigram, NO log-transform) ===
t0 = time.time()
lr_a = LinearRegression()
lr_a.fit(X_a_train, train_prices)  # train trên raw VND
print(f"LR Arch A train: {time.time()-t0:.1f}s")

pred_lr_a = np.clip(lr_a.predict(X_a_test), 0, None)  # clip âm về 0
results["LR + Arch A"] = plot_predictions(
    test_prices, pred_lr_a,
    title="LR + Arch A (TF-IDF bigram, raw, no word segmentation)",
    names=test_names, plot_size=PLOT_SIZE,
)

# === LR + Arch B (Underthesea word tokenize, NO log-transform) ===
t0 = time.time()
lr_b = LinearRegression()
lr_b.fit(X_b_train, train_prices)  # train trên raw VND
print(f"LR Arch B train: {time.time()-t0:.1f}s")

pred_lr_b = np.clip(lr_b.predict(X_b_test), 0, None)
results["LR + Arch B"] = plot_predictions(
    test_prices, pred_lr_b,
    title="LR + Arch B (TF-IDF + Underthesea word segmentation, raw)",
    names=test_names, plot_size=PLOT_SIZE,
)

LR Arch A train: 11.6s


LR Arch B train: 2.8s


## 7. Tier 3 — ML Models: Log-transform + Category Feature

### 7a. Ridge Regression (RMSLE = 0.5415)

**Ridge** = Linear Regression + L2 regularization: `loss = MSE + alpha * ||w||²`

- `alpha=1.0`: penalize weight lớn → tránh overfitting trên 10,008 sparse features
- Train chỉ **0.6 giây** — nhanh nhất trong tất cả models
- Train **trên `log1p(price)`** → predict **`expm1(pred)`** → không bao giờ predict âm
- RMSLE giảm từ 1.495 (LR raw) → 0.5415 — log-transform cải thiện **64%**
- Kém LGB default chỉ 2.4% → L2 regularization + log-transform đã khai thác gần hết thông tin từ TF-IDF


In [10]:
# === Ridge + Arch B + Cat (log-transform) ===
# Arch B + Cat = TF-IDF Underthesea (10K) + OneHot Category (8) = 10,008 features
t0 = time.time()
ridge = Ridge(alpha=1.0)
ridge.fit(X_bc_train, log_train_prices)  # train trên log1p(price)
print(f"Ridge (B+Cat, log) train: {time.time()-t0:.1f}s")

pred_ridge = np.clip(np.expm1(ridge.predict(X_bc_test)), 0, None)
results["Ridge + Arch B+Cat"] = plot_predictions(
    test_prices, pred_ridge,
    title="Ridge + Arch B+Cat (alpha=1.0, log1p target)",
    names=test_names, plot_size=PLOT_SIZE,
)

Ridge (B+Cat, log) train: 0.6s


### 7b. LightGBM Default (RMSLE = 0.5286)

**LightGBM** (Gradient Boosting Decision Tree) — nhanh nhất trong họ GBDT nhờ:
- **Leaf-wise tree growth** (thay vì level-wise của XGBoost): tìm leaf có gain lớn nhất để split
- **Histogram-based splitting**: bin continuous features → tìm split điểm trong O(bins) thay O(N)

Config default:
- `n_estimators=1000`, `learning_rate=0.1`, `n_jobs=-1`
- Train **37.8 giây** trên 85,727 x 10,008 sparse features
- Cải thiện 1.3% so với Ridge (0.5415 → 0.5286) nhờ phi tuyến tính


In [11]:
# === LightGBM + Arch B + Cat (log-transform, default params) ===
t0 = time.time()
lgb_default = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
lgb_default.fit(X_bc_train, log_train_prices)
print(f"LightGBM (B+Cat, log) train: {time.time()-t0:.1f}s")

pred_lgb_default = np.clip(np.expm1(lgb_default.predict(X_bc_test)), 0, None)
results["LightGBM + Arch B+Cat"] = plot_predictions(
    test_prices, pred_lgb_default,
    title="LightGBM + Arch B+Cat (default, log1p target)",
    names=test_names, plot_size=PLOT_SIZE,
)

LightGBM (B+Cat, log) train: 37.8s


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



### 7c. LightGBM Tuned + Arch C + Cat (RMSLE = 0.5231)

Best params từ Optuna 50 trials — hardcode để tái sử dụng (không cần chạy lại 1h28m):

| Param | Giá trị | Ý nghĩa |
|-------|---------|----------|
| `num_leaves` | 173 | Độ phức tạp cây — càng cao càng fit train tốt hơn |
| `min_child_samples` | 50 | Số mẫu tối thiểu mỗi leaf — regularize |
| `feature_fraction` | 0.689 | Dùng 68.9% features mỗi cây — giảm correlation giữa cây |
| `lambda_l1` | 0.011 | L1 regularization — sparsity |
| `lambda_l2` | 0.155 | L2 regularization — smooth weights |
| `learning_rate` | 0.032 | Nhỏ hơn default (0.1) → generalize tốt hơn |
| `n_estimators` | 1500 | Tăng từ 1000 vì lr nhỏ hơn |

**Arch C + Cat** = `FeatureUnion(word bigram 5K + char_wb 3-5gram 5K)` + OneHot Category (8) = **10,008 features**  
char_wb bắt được `"samsung"`, `"128gb"`, `"iphone_14"` dù viết không dấu.  
**Train time: ~20 phút** (1,205 giây) — cần nhiều thời gian hơn vì `learning_rate=0.032` x 1,500 cây.


In [12]:
# === LightGBM Tuned + Arch C + Cat ===
# Best params từ Optuna 50 trials (day3 v3, val RMSLE=0.5588)
BEST_LGB_PARAMS = {
    "num_leaves":        173,
    "min_child_samples": 50,
    "feature_fraction":  0.689,
    "lambda_l1":         0.011,
    "lambda_l2":         0.155,
    "learning_rate":     0.032,
    "n_estimators":      1500,
    "random_state":      SEED,
    "n_jobs":            -1,
    "verbose":           -1,
}

t0 = time.time()
lgb_tuned = lgb.LGBMRegressor(**BEST_LGB_PARAMS)
lgb_tuned.fit(X_cc_train, log_train_prices)
print(f"LightGBM Tuned (C+Cat, log) train: {time.time()-t0:.1f}s")

pred_lgb_tuned = np.clip(np.expm1(lgb_tuned.predict(X_cc_test)), 0, None)
results["LightGBM Tuned + Arch C+Cat"] = plot_predictions(
    test_prices, pred_lgb_tuned,
    title="LightGBM Tuned + Arch C+Cat (Optuna best params, log1p target)",
    names=test_names, plot_size=PLOT_SIZE,
)

LightGBM Tuned (C+Cat, log) train: 1205.2s


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



## 8. Tier 4 — Weighted Blending

Kết hợp predictions từ 3 model tốt nhất. Dùng `scipy.optimize.minimize` (Nelder-Mead)  
tìm weights tối ưu trên **validation set**, sau đó áp dụng lên test set.

**Weights tìm được (val RMSLE tối ưu = 0.5209):**

| Model | Weight |
|-------|-------:|
| LightGBM Tuned C+Cat | 0.543 |
| Ridge B+Cat | 0.263 |
| LightGBM B+Cat | 0.195 |

**Nhận xét:** Ridge được cấp 26.3% weight — đây là điều bất ngờ.  
Ridge có error pattern khác LGB (linear vs non-linear) → diversity thực sự có giá trị cho blending.

**Tại sao blending chỉ cải thiện nhỏ (0.5231 → 0.5088, -1.4%)?**  
Ba models này đều dùng TF-IDF sparse cùng data → correlation cao → diversity thấp.  
Blending hiệu quả nhất khi models có error pattern khác nhau — Day 4 giải quyết bằng dense embeddings.


In [13]:
# Predictions trên val set để tìm weights tối ưu
pred_lgb_tuned_val  = np.clip(np.expm1(lgb_tuned.predict(X_cc_val)),   0, None)
pred_lgb_default_val = np.clip(np.expm1(lgb_default.predict(X_bc_val)), 0, None)
pred_ridge_val      = np.clip(np.expm1(ridge.predict(X_bc_val)),        0, None)

val_preds_list = [pred_lgb_tuned_val, pred_lgb_default_val, pred_ridge_val]
blend_names    = ["LightGBM Tuned C+Cat", "LightGBM B+Cat", "Ridge B+Cat"]

def blend_rmsle(weights):
    w = np.abs(weights)
    w = w / w.sum()
    blended = sum(wi * p for wi, p in zip(w, val_preds_list))
    return rmsle(val_prices, blended)

result_opt = scipy_minimize(blend_rmsle, x0=[0.5, 0.3, 0.2], method="Nelder-Mead")
best_weights = np.abs(result_opt.x)
best_weights /= best_weights.sum()

print("Optimal weights (minimized on val set):")
for name, w in zip(blend_names, best_weights):
    print(f"  {name}: {w:.3f}")
print(f"Val blend RMSLE: {result_opt.fun:.4f}")

# Áp dụng weights lên test set
test_preds_list = [pred_lgb_tuned, pred_lgb_default, pred_ridge]
pred_blended = sum(w * p for w, p in zip(best_weights, test_preds_list))

results["Blended (3 models)"] = plot_predictions(
    test_prices, pred_blended,
    title="Blended: LGB Tuned C + LGB B + Ridge (weights from val)",
    names=test_names, plot_size=PLOT_SIZE,
)

c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



Optimal weights (minimized on val set):
  LightGBM Tuned C+Cat: 0.543
  LightGBM B+Cat: 0.195
  Ridge B+Cat: 0.263
Val blend RMSLE: 0.5209


## 9. Final Results

Toàn bộ metrics tính trên **3,872 test items** (full test set, không phải 200).  
Bảng sắp xếp theo RMSLE tăng dần.

In [14]:
print("=" * 80)
print(f"FINAL RESULTS — Day 3 Final (Test: {len(test):,} items, <= {PRICE_THRESHOLD:,} VND)")
print("=" * 80)
print(f"\n{'Model':<35} {'RMSLE':>8} {'MAE (VND)':>13} {'MAPE':>8} {'R2':>7}")
print("-" * 78)

sorted_results = sorted(results.items(), key=lambda x: x[1]["rmsle"])
for name, res in sorted_results:
    print(
        f"{name:<35} {res['rmsle']:>8.4f} {res['mae']:>13,.0f} "
        f"{res['mape']:>7.1f}% {res['r2']:>6.1f}%"
    )

best_name, best_res = sorted_results[0]
print(f"\nBest model: {best_name} — RMSLE={best_res['rmsle']:.4f}")

# So sánh log-transform impact
lr_b_rmsle     = results["LR + Arch B"]["rmsle"]
lgb_best_rmsle = best_res["rmsle"]
print(f"\nLog-transform + GBDT vs LR raw:")
print(f"  LR + Arch B (no log): {lr_b_rmsle:.4f}")
print(f"  {best_name}: {lgb_best_rmsle:.4f}")
print(f"  Improvement: {(lr_b_rmsle - lgb_best_rmsle) / lr_b_rmsle * 100:.1f}%")

print("\n" + "=" * 80)
print("Day 3 ceiling: RMSLE ~ 0.52 (TF-IDF bag-of-words limit)")
print("Day 4: Dense embeddings (PhoBERT, AITeamVN) -> target RMSLE <= 0.40")
print("=" * 80)

FINAL RESULTS — Day 3 Final (Test: 3,872 items, <= 1,000,000 VND)

Model                                  RMSLE     MAE (VND)     MAPE      R2
------------------------------------------------------------------------------
Blended (3 models)                    0.5088       108,181    43.6%   48.8%
LightGBM Tuned + Arch C+Cat           0.5231       110,429    44.2%   46.1%
LightGBM + Arch B+Cat                 0.5286       112,889    45.1%   44.7%
Ridge + Arch B+Cat                    0.5415       114,785    46.6%   42.5%
Median                                0.7736       169,571    77.4%  -10.4%
Mean                                  0.8219       180,084   106.2%   -0.0%
Random                                1.2970       340,331   226.0% -232.1%
LR + Arch B                           1.4950       122,215    62.2%   47.0%
LR + Arch A                           1.5216       124,523    63.7%   45.4%

Best model: Blended (3 models) — RMSLE=0.5088

Log-transform + GBDT vs LR raw:
  LR + Arch B 

## 10. Phân tích & Bài học

### Kết quả cuối cùng

| Model | RMSLE | MAE (VND) | MAPE | R2 |
|-------|------:|----------:|-----:|---:|
| **Blended (3 models)** | **0.5088** | **108,181** | **43.6%** | **48.8%** |
| LightGBM Tuned C+Cat | 0.5231 | 110,429 | 44.2% | 46.1% |
| LightGBM B+Cat | 0.5286 | 112,889 | 45.1% | 44.7% |
| Ridge B+Cat | 0.5415 | 114,785 | 46.6% | 42.5% |
| Median | 0.7736 | 169,571 | 77.4% | -10.4% |
| Mean | 0.8219 | 180,084 | 106.2% | 0.0% |
| Random | 1.2970 | 340,331 | 226.0% | -232.1% |
| LR + Arch B | 1.4950 | 122,215 | 62.2% | 47.0% |
| LR + Arch A | 1.5216 | 124,523 | 63.7% | 45.4% |

### Impact của từng cải tiến

| Cải tiến | RMSLE trước → sau | % giảm | Impact |
|----------|-------------------|--------|--------|
| **Log-transform** (LR B raw → Ridge log) | 1.4950 → 0.5415 | **-63.8%** | **Lớn nhất — bắt buộc** |
| **GBDT thay Linear** (Ridge → LGB default) | 0.5415 → 0.5286 | -2.4% | Trung bình |
| **Arch C + Optuna tuning** (LGB B → LGB Tuned C) | 0.5286 → 0.5231 | -1.0% | Nhỏ |
| **Blending 3 models** | 0.5231 → 0.5088 | -1.4% | Nhỏ |

### Bài học chính

1. **Log-transform bắt buộc với RMSLE** — không log → predict âm → clip 0 → sai lầm tệ hơn đoán mò.
2. **RMSLE ≠ MAE/R2** — LR có MAE tốt, R2 tốt, nhưng RMSLE thảm họa. Mỗi metric đo một khía cạnh khác.
3. **Ridge cạnh tranh với LGB** trên TF-IDF sparse (chỉ kém 2.4%) — đáng ngạc nhiên và train chỉ 0.6s.
4. **Blending có ích khi diversity cao** — ở đây diversity thấp (cùng TF-IDF) nên chỉ -1.4%.
5. **Ceiling TF-IDF ~0.52** — mất thứ tự từ, ngữ nghĩa, ngữ cảnh → cần dense embeddings.

### Hướng Day 4

| Approach | Model | Actual RMSLE |
|----------|-------|--------------|
| Frozen embeddings | AITeamVN (BGE-M3 1024d) + MLP | 0.4986 |
| Full fine-tune | PhoBERT-base-v2 (135M) | 0.4418 |
| Fine-tune + LLRD + EMA + R-Drop | PhoBERT++ | 0.4322 |
| Stacking 7 models | Ridge + ElasticNet + LGB meta | **0.4059** |
